In [5]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset (Alzheimer's Disease)
# ----------------------------------------------------
candidate_paths = [
    Path("Alzhimers.xlsx"),
    Path("Alzheimer.xlsx"),
    Path("clean_alzheimer.csv"),
    Path("../Other GANS/clean_alzheimer.csv"),
    Path("../../Other GANS/clean_alzheimer.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Alzheimer dataset not found. Place Alzhimers.xlsx or clean_alzheimer.csv in this folder."
    )

if data_path.suffix.lower() in {".xlsx", ".xls"}:
    raw_data = pd.read_excel(data_path)
else:
    raw_data = pd.read_csv(data_path)

target_col = "Group"
ad_data = raw_data.drop(columns=["Subject ID", "M/F", "MRI ID", "Hand"], errors="ignore")
ad_data[target_col] = ad_data[target_col].replace({"Demented": 1, "Nondemented": 0, "Converted": 1})
ad_data[target_col] = pd.to_numeric(ad_data[target_col], errors="coerce").fillna(0).astype(int)

for col in ad_data.select_dtypes(include=[np.number]).columns:
    if ad_data[col].isnull().any():
        ad_data[col] = ad_data[col].fillna(ad_data[col].mean())

for col in ad_data.select_dtypes(include=["object"]).columns:
    if ad_data[col].isnull().any():
        modes = ad_data[col].mode()
        ad_data[col] = ad_data[col].fillna(modes[0] if len(modes) else "")

# Use same variable name pattern as other notebooks
alzheimer_data = ad_data.copy()

X = alzheimer_data.drop(columns=[target_col])
y = alzheimer_data[target_col]

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(alzheimer_data)

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []

In [7]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ---------------------------------------------------
# TRAIN / TEST SPLIT (NO LEAKAGE)
# ---------------------------------------------------

train_real, test_real = train_test_split(
    alzheimer_data,
    test_size=TEST_SIZE,
    stratify=alzheimer_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)


================ SINGLE RUN ================
Training TabDDPM...
[0]
12
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(12)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.4178 Sum: 0.4178
Step 1000/1000 MLoss: 0.0 GLoss: 0.3634 Sum: 0.3634
mlp
Sample timestep    0
Discrete cols: [0, 3]
Num shape:  (1000, 10)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 455.56it/s]|
Column Shapes Score: 63.38%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 290.38it/s]|
Column Pair Trends Score: 44.27%

Overall Score (Average): 53.82%

TabDDPM: 0.5382


In [8]:
# ForestDiffusion

try:
    import traceback

    print("Training ForestDiffusion...")
    synthetic_forestdiffusion = train_forestdiffusion(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_forestdiffusion,
        metadata=train_metadata,
    )

    scores["ForestDiffusion"] = quality.get_score()

    print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))

    del synthetic_forestdiffusion

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("ForestDiffusion Failed:")
    traceback.print_exc()

Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 428.26it/s]|
Column Shapes Score: 72.66%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 255.49it/s]|
Column Pair Trends Score: 46.59%

Overall Score (Average): 59.63%

ForestDiffusion: 0.5963


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': LinearSVC(max_iter=2000, dual='auto', random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}


In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51]
):

    results = []
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[label_col] = pd.to_numeric(train_df[label_col], errors="coerce").astype(int)
    test_df[label_col] = pd.to_numeric(test_df[label_col], errors="coerce").astype(int)

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=y_train
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=y_test
            )

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )


In [17]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = "Group"

model_order = ["TabDDPM", "ForestDiffusion"]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=alzheimer_data,
    test_df=alzheimer_data,
    label_col="Group",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not trained")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=alzheimer_data,
        label_col="Group",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
1,SVM-RBF,0.9573 ± 0.0196,0.9549 ± 0.0216,0.9886 ± 0.0139,0.9243 ± 0.0378
0,LogReg,0.9560 ± 0.0224,0.9533 ± 0.0251,0.9886 ± 0.0139,0.9216 ± 0.0443
3,NaiveBayes,0.9547 ± 0.0171,0.9527 ± 0.0185,0.9776 ± 0.0163,0.9297 ± 0.0324
5,RandomForest,0.9533 ± 0.0137,0.9514 ± 0.0149,0.9752 ± 0.0190,0.9297 ± 0.0324
8,AdaBoost,0.9493 ± 0.0259,0.9471 ± 0.0281,0.9674 ± 0.0286,0.9297 ± 0.0501
6,ExtraTrees,0.9480 ± 0.0163,0.9457 ± 0.0181,0.9696 ± 0.0191,0.9243 ± 0.0378
7,GradientBoost,0.9453 ± 0.0219,0.9434 ± 0.0229,0.9621 ± 0.0315,0.9270 ± 0.0383
9,MLP,0.9387 ± 0.0281,0.9369 ± 0.0301,0.9485 ± 0.0325,0.9270 ± 0.0453
2,KNN,0.9160 ± 0.0292,0.9106 ± 0.0352,0.9513 ± 0.0446,0.8784 ± 0.0676
4,DecisionTree,0.9067 ± 0.0458,0.9092 ± 0.0424,0.8895 ± 0.0692,0.9324 ± 0.0277


TabDDPM - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
8,AdaBoost,0.5880 ± 0.0781,0.4953 ± 0.0953,0.6319 ± 0.1347,0.4108 ± 0.0853
3,NaiveBayes,0.5693 ± 0.0396,0.2819 ± 0.1273,0.8424 ± 0.1718,0.1838 ± 0.1024
7,GradientBoost,0.5693 ± 0.0382,0.4665 ± 0.0669,0.5977 ± 0.0588,0.3865 ± 0.0736
5,RandomForest,0.5373 ± 0.0426,0.4339 ± 0.0633,0.5460 ± 0.0652,0.3622 ± 0.0664
6,ExtraTrees,0.5213 ± 0.0660,0.4143 ± 0.0995,0.5159 ± 0.0987,0.3486 ± 0.0970
0,LogReg,0.5200 ± 0.0396,0.1648 ± 0.0940,0.6544 ± 0.2493,0.1000 ± 0.0605
2,KNN,0.5200 ± 0.0422,0.4672 ± 0.0426,0.5185 ± 0.0504,0.4270 ± 0.0449
1,SVM-RBF,0.5187 ± 0.0398,0.1653 ± 0.0880,0.6572 ± 0.2398,0.1000 ± 0.0568
4,DecisionTree,0.5093 ± 0.0213,0.4917 ± 0.0455,0.5019 ± 0.0214,0.4865 ± 0.0764
9,MLP,0.4667 ± 0.0520,0.3812 ± 0.0675,0.4459 ± 0.0744,0.3351 ± 0.0686


TabDDPM - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,0.9653 ± 0.0136,0.9640 ± 0.0143,0.9860 ± 0.0140,0.9432 ± 0.0225
9,MLP,0.9627 ± 0.0205,0.9614 ± 0.0212,0.9781 ± 0.0236,0.9459 ± 0.0320
8,AdaBoost,0.9613 ± 0.0139,0.9597 ± 0.0150,0.9835 ± 0.0179,0.9378 ± 0.0297
7,GradientBoost,0.9613 ± 0.0173,0.9594 ± 0.0186,0.9887 ± 0.0138,0.9324 ± 0.0325
5,RandomForest,0.9613 ± 0.0183,0.9594 ± 0.0198,0.9887 ± 0.0138,0.9324 ± 0.0347
0,LogReg,0.9573 ± 0.0196,0.9549 ± 0.0216,0.9886 ± 0.0139,0.9243 ± 0.0378
1,SVM-RBF,0.9573 ± 0.0196,0.9549 ± 0.0216,0.9886 ± 0.0139,0.9243 ± 0.0378
3,NaiveBayes,0.9507 ± 0.0198,0.9488 ± 0.0210,0.9697 ± 0.0251,0.9297 ± 0.0324
2,KNN,0.9467 ± 0.0304,0.9426 ± 0.0342,0.9882 ± 0.0145,0.9027 ± 0.0582
4,DecisionTree,0.9187 ± 0.0249,0.9170 ± 0.0264,0.9219 ± 0.0304,0.9135 ± 0.0415


ForestDiffusion - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,ForestDiffusion,SVM-RBF,0.000000,0.000000,0.000000,0.000000,0.9573 ± 0.0196,0.9573 ± 0.0196
1,ForestDiffusion,LogReg,-0.001333,-0.001624,0.000000,-0.002703,0.9560 ± 0.0224,0.9573 ± 0.0196
2,ForestDiffusion,NaiveBayes,0.004000,0.003848,0.007901,0.000000,0.9547 ± 0.0171,0.9507 ± 0.0198
3,ForestDiffusion,RandomForest,-0.008000,-0.007995,-0.013517,-0.002703,0.9533 ± 0.0137,0.9613 ± 0.0183
4,ForestDiffusion,AdaBoost,-0.012000,-0.012579,-0.016096,-0.008108,0.9493 ± 0.0259,0.9613 ± 0.0139
5,ForestDiffusion,ExtraTrees,-0.017333,-0.018266,-0.016458,-0.018919,0.9480 ± 0.0163,0.9653 ± 0.0136
6,ForestDiffusion,GradientBoost,-0.016000,-0.015983,-0.026614,-0.005405,0.9453 ± 0.0219,0.9613 ± 0.0173
7,ForestDiffusion,MLP,-0.024000,-0.024528,-0.029636,-0.018919,0.9387 ± 0.0281,0.9627 ± 0.0205
8,ForestDiffusion,KNN,-0.030667,-0.031937,-0.036927,-0.024324,0.9160 ± 0.0292,0.9467 ± 0.0304
9,ForestDiffusion,DecisionTree,-0.012000,-0.007818,-0.032430,0.018919,0.9067 ± 0.0458,0.9187 ± 0.0249


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,-0.011733,-0.011688,-0.016378,-0.006216
1,TabDDPM,0.410533,0.564317,0.370660,0.608378


In [18]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
